# 操作系统

> 面试定位：**进程/线程/协程、调度、同步互斥、死锁、虚拟内存、零拷贝** 是后端岗必考。
> 标记：🔴 必背 / 🟡 掌握 / 🟢 了解。本 notebook 配套 `02-操作系统.md` 速查文档，全部代码可运行。


## 进程 vs 线程 vs 协程


![os_process_states.png](images/os_process_states.png)

*配图来源：本仓库 matplotlib 绘制（`images/os_process_states.png`）*


### 进程

**进程** 是操作系统资源分配的基本单位，每个进程拥有独立的**虚拟地址空间**、**堆栈**、**文件描述符表**等资源。

**资源占用**：高（几MB到数百MB）
**切换成本**：高（上下文切换涉及内核态切换、TLB刷新等）
**通信方式**：通过IPC（如管道、消息队列、共享内存）
**适用场景**：需要隔离的场景（如浏览器多标签、游戏多开）、长时间运行的任务


### 线程

**线程** 是进程内部的一个执行流，共享进程的地址空间和资源。

**资源占用**：低（几KB到几MB）
**切换成本**：中（共享内存，切换较快）
**通信方式**：直接读写共享内存
**适用场景**：需要并发处理任务、I/O密集型（如网络爬虫、服务器处理多个连接）


### 协程

**协程** 是用户态的轻量级线程，由程序自行调度。

**资源占用**：极低（几十到几百字节）
**切换成本**：极低（无内核切换）
**通信方式**：通过通道或共享内存
**适用场景**：高并发、少量阻塞的任务（如异步IO、爬虫协程模式）


### 对比表格

| 特性       | 进程         | 线程         | 协程         |
|------------|--------------|--------------|--------------|
| 资源占用   | 最高         | 中等         | 最低         |
| 切换成本   | 最高         | 中等         | 最低         |
| 通信方式   | IPC          | 共享内存     | 通道/共享内存|
| 调度者     | 内核         | 内核/用户     | 用户         |
| 内存隔离   | 有           | 无           | 无           |
| 适用场景   | 隔离需求     | 并发任务     | 高并发轻量任务|


### 易错点/面试点评
- **易混**：很多人把协程和线程等同，其实协程是用户态调度，无内核参与。
- **面试点**：面试官常追问“为什么协程比线程更轻量？”答案：无内核态切换，调度在用户空间。

---


## 进程状态转换图与调度算法


### 进程状态转换图

进程状态包括：
- **新建**：进程被创建
- **就绪**：进程具备运行条件，等待CPU
- **运行**：进程占有CPU正在执行
- **阻塞/等待**：进程等待某事件（如I/O）
- **结束/退出**：进程完成或被终止


### 调度算法

#### FCFS（先来先服务）
- **原理**：按进程到达顺序排队
- **例子**：进程P1(0,8), P2(1,4), P3(2,9) → 平均等待时间 = (0+1+2) = 3
- **优缺点**：公平，但短作业可能等待长作业

#### SJF/SRTF（短作业优先）
- **原理**：选择剩余时间最短的进程
- **例子**：P1(6,2), P2(2,5), P3(4,4) → 平均等待时间 = 5
- **优缺点**：短作业好，但可能饿死长作业

#### 时间片轮转（RR）
- **原理**：每个进程分配固定时间片，时间到换下
- **例子**：3个进程，每个时间片2单位 → 总等待时间计算
- **优缺点**：响应快，适合分时系统

#### 优先级调度
- **原理**：按优先级分配CPU
- **例子**：优先级1(1,10), 优先级2(2,5) → 等待时间
- **优缺点**：高优先级可能饿死低优先级

#### 多级反馈队列
- **原理**：多级队列，时间片递减，优先级递减
- **例子**：队列0(时间片8)，队列1(时间片4)，队列2(时间片2)
- **优缺点**：平衡短作业和长作业


### 对比表格

| 算法         | 平均等待时间 | 吞吐量 | 响应时间 | 适合场景       |
|--------------|--------------|--------|----------|----------------|
| FCFS         | 中等         | 低     | 差       | 批处理         |
| SJF          | 低           | 中     | 好       | 短作业         |
| RR           | 中等         | 中     | 好       | 分时系统       |
| 优先级       | 视情况       | 视情况 | 视情况   | 实时系统       |


### 面试追问
- **追问**：如何避免短作业饿死？→ 多级反馈队列
- **标准回答**：为长作业设置最小时间片，优先调度短作业。

---


## 同步与互斥


### 同步与互斥

- **互斥**：多个进程/线程不能同时访问临界资源
- **同步**：进程/线程间按特定顺序执行


### 关键概念

- **临界区**：访问共享资源的代码段
- **互斥锁**：保证只有一个进程进入临界区
- **自旋锁**：忙等待，适合短临界区
- **读写锁**：区分读和写
- **条件变量**：线程等待特定条件
- **信号量(P/V)**：计数信号量


### 生产者-消费者代码示例（Python）


In [ ]:
import threading
import time

buffer = []
MAX_SIZE = 5
lock = threading.Lock()
empty = threading.Semaphore(MAX_SIZE)
full = threading.Semaphore(0)

def producer():
    for i in range(10):
        empty.acquire()
        lock.acquire()
        buffer.append(i)
        print(f"生产者生产: {i}")
        lock.release()
        full.release()
        time.sleep(0.05)

def consumer():
    for i in range(10):
        full.acquire()
        lock.acquire()
        item = buffer.pop(0)
        print(f"消费者消费: {item}")
        lock.release()
        empty.release()
        time.sleep(0.05)

t1 = threading.Thread(target=producer)
t2 = threading.Thread(target=consumer)
t1.start()
t2.start()
t1.join()
t2.join()



### 对比表格

| 机制     | 实现方式     | 适用场景         |
|----------|--------------|------------------|
| 互斥锁   | acquire/release | 单一资源访问     |
| 自旋锁   | while lock.acquire() | 短临界区         |
| 读写锁   | reader/writer | 读多写少         |
| 信号量   | P/V 操作     | 计数同步         |
| 条件变量 | wait/signal  | 复杂条件等待     |


### 易错点/面试点评
- **易错**：信号量P/V操作顺序错误会导致死锁。
- **面试点**：面试官常问“生产者消费者模型如何实现？”答案：用信号量或锁+条件变量。

---


## 死锁


### 四个必要条件

1. **互斥使用**：资源只能被一个进程使用
2. **不可剥夺**：资源不能被抢占
3. **保持并等待**：已获得资源的进程仍需资源
4. **循环等待**：资源分配形成环形等待


### 死锁避免

**银行家算法**：
- 安全状态检查：是否存在分配序列使所有进程都能完成
- 例子：3个进程，资源3、2、2
  - 安全序列：P1(1,0,0) -> P3(0,1,0) -> P2(0,0,1)


### 死锁检测与解除

- **检测**：资源分配图，查找环
- **解除**：剥夺资源或终止进程
- **破坏条件**：破坏4个条件之一


### 对比表格

| 方法       | 优点           | 缺点             |
|------------|----------------|------------------|
| 避免       | 防止死锁发生   | 需提前知道资源请求 |
| 检测+解除 | 允许死锁发生   | 需要检测和恢复   |


### 面试追问
- **追问**：如何检测死锁？→ 资源分配图
- **标准回答**：用Banker's Algorithm避免死锁。

---


## 进程间通信 IPC


### 对比表格

| 方式       | 通信方式     | 速度     | 适用场景           | 内核拷贝 |
|------------|--------------|----------|--------------------|----------|
| 管道       | 半双工       | 低       | 亲缘进程           | 有       |
| 消息队列   | 消息传递     | 中       | 任意进程           | 有       |
| 共享内存   | 直接读写     | 最高     | 高速通信           | 无       |
| 信号量     | 计数同步     | 高       | 同步              | 无       |
| 信号       | 信号传递     | 快       | 通知进程           | 无       |
| Socket     | 网络通信     | 低       | 不同主机           | 有       |


### 易错点/面试点评
- **易错**：管道是半双工，需注意读写顺序。
- **面试点**：问IPC机制选择，答案：共享内存适合高速，Socket适合网络。

---


## 虚拟内存


![os_paging.png](images/os_paging.png)

*配图来源：本仓库 matplotlib 绘制（`images/os_paging.png`）*


### 分页与页表

- **分页**：将内存分成固定大小页
- **页表**：记录页号到物理页号的映射


### 缺页中断流程

1. CPU访问页不在内存
2. 触发缺页中断
3. 缺页中断处理程序加载页
4. 恢复进程


### 页面置换算法

- **FIFO**：简单，但可能Belady异常
- **LRU**：最近最少使用，效果好
- **OPT**：最佳，但需未来信息
- **Clock**：时钟算法，近似LRU


### LRU Python实现


In [ ]:
class LRUCache:
    def __init__(self, capacity):
        self.capacity = capacity
        self.cache = {}
        self.order = []  # 使用list模拟顺序
    
    def get(self, key):
        if key not in self.cache:
            return -1
        # 移动到末尾
        val = self.cache.pop(key)
        self.cache[key] = val
        self.order.remove(key)
        self.order.append(key)
        return val
    
    def put(self, key, value):
        if key in self.cache:
            self.order.remove(key)
            self.cache.pop(key)
        elif len(self.cache) >= self.capacity:
            lru = self.order.pop(0)
            self.cache.pop(lru)
        self.cache[key] = value
        self.order.append(key)


### 对比表格

| 算法 | 优点         | 缺点             | Belady异常 |
|------|--------------|------------------|------------|
| FIFO | 简单         | Belady异常       | 是         |
| LRU  | 性能好       | 需要维护顺序     | 否         |
| OPT  | 最佳         | 需要未来信息     | 否         |
| Clock| 近似LRU      | 实现复杂         | 否         |


### 易错点/面试点评
- **易错**：LRU实现时需注意顺序维护。
- **面试点**：问页表结构，答案：多级页表减少内存占用。

---


## 用户态 vs 内核态


### 为什么区分

- **内核态**：执行内核代码，有最高权限
- **用户态**：执行用户代码，受限


### 系统调用流程

1. 用户态调用陷入**trap**
2. CPU切换到内核态
3. 内核执行系统调用
4. 返回用户态


### 易错点/面试点评
- **易错**：系统调用不是函数调用，是陷阱机制。
- **面试点**：问特权指令，答案：在用户态不能执行。

---


## 零拷贝


### 原理

- **mmap**：将文件映射到内存，避免拷贝
- **sendfile**：内核直接传输文件到Socket
- **DMA**：直接内存访问，CPU不参与拷贝


### 写时复制（COW）

- 共享内存页，当写入时才复制


### 易错点/面试点评
- **面试点**：问零拷贝场景，答案：大文件传输。

---


## 常见问题


### 僵尸进程 vs 孤儿进程

- **僵尸**：子进程结束，父未调用wait
- **孤儿**：父进程结束，子被init接管


### 上下文切换成本

- 寄存器、页表、TLB刷新等


### 线程池大小

- CPU核心数 * 2 或 I/O密集型 * (1 + 阻塞时间/计算时间)


### 易错点/面试点评
- **面试点**：问线程池参数，答案：根据业务调整。

---


## 调度算法算例详解（面试手算题）


### GPA 计算示例：FCFS vs SJF vs RR（背下这套手算）

设 3 个进程到达时间与运行时间如下：

| 进程 | 到达时间 | 运行时间 |
|------|----------|----------|
| P1 | 0 | 8 |
| P2 | 1 | 4 |
| P3 | 2 | 9 |

**FCFS（先来先服务）**：按到达顺序执行 P1→P2→P3

```
时间轴: |----P1(0-8)----|--P2(8-12)--|-----P3(12-21)-----|
```

- 等待时间：P1=0，P2=8-1=7，P3=12-2=10
- **平均等待时间 = (0+7+10)/3 ≈ 5.67**，平均周转时间 = ((8-0)+(12-1)+(21-2))/3 = 38/3 ≈ 12.67

**SJF（非抢占短作业优先）**：0 时刻只有 P1 到达 → 先跑 P1（8 个时间单位）；期间 P2(1)、P3(2) 已到达，P1 结束后选最短的 P2：

```
时间轴: |----P1(0-8)----|--P2(8-12)--|-----P3(12-21)-----|
```

- 等待时间：P1=0，P2=8-1=7，P3=12-2=10（本例与 FCFS 相同，因 P1 先到且最长）
- **注意**：SJF 的收益在"短作业先于长作业"时最明显：若 P3 运行时间改为 3、P2 为 9，则 P1 结束后应选 P3 → 平均等待显著下降

**RR（时间片 q=3）**：就绪队列轮转

```
时间轴: |P1(0-3)|P2(3-6)|P3(6-9)|P1(9-12)|P2(12-13)|P3(13-16)|P1(16-21)|
```

- 等待时间：P1=6+4+5=15？——按"首次就绪到首次调度"简化计算：P1 等待 = 3+3+3=9，P2 等待 = 4，P3 等待 = 5
- **平均等待 = (9+4+5)/3 = 6**，响应时间最优（每进程最多等 2q）

**手算结论（背）**：FCFS 公平但长作业拖垮短作业；SJF 平均等待最小但**可能饿死长作业**且**不可抢占**；RR 响应快适合分时系统，时间片太小切换开销大、太大退化成 FCFS。


### 多级反馈队列（MLFQ，Linux CFS 之前的经典）

```
Q0 (时间片 8ms, RR):   高优先级，新进程进来
Q1 (时间片 16ms, RR):  从 Q0 用满降级
Q2 (时间片 32ms, RR):  从 Q1 用满降级
Q3 (FCFS 或 RR):       后台批处理
```

- **短作业在高层快速完成，长作业逐步降级**——同时兼顾响应时间与吞吐
- **问题**：可能导致饥饿（老长作业一直轮不到）→ 通常加"每过一段时间把所有进程升回 Q0"或"按等待时间提升优先级"

---


## 银行家算法完整算例（死锁避免手算）

**场景**：3 个进程 P0/P1/P2，系统资源总量 `A=3, B=3, C=2`。

分配矩阵 `Allocation`、最大需求 `Max`：

| 进程 | Alloc(A,B,C) | Max(A,B,C) | Need = Max-Alloc |
|------|--------------|------------|------------------|
| P0 | 0,1,0 | 1,2,2 | 1,1,2 |
| P1 | 2,0,0 | 4,0,2 | 2,0,2 |
| P2 | 3,0,2 | 4,1,2 | 1,1,0 |

**可用资源 Available** = 总量 − 已分配 = `(3,3,2) − (5,1,2) = (1,2,0)`。注意 **C 已耗尽**。

**安全性检查（依次找"Need ≤ Available"的进程）**：

1. P0 的 Need=(1,1,2)：需要 C=2 > Available 的 C=0 → **不满足**
2. P1 的 Need=(2,0,2)：C=2 > 0 → 不满足
3. P2 的 Need=(1,1,0)：(1,1,0) ≤ (1,2,0) → **满足**，先执行 P2；P2 释放后 Available = (1,2,0)+(3,0,2) = (4,2,2)
4. 现在 P1 的 Need=(2,0,2) ≤ (4,2,2) → 满足，执行 P1；释放后 Available = (4,2,2)+(2,0,0) = (6,2,2)
5. P0 的 Need=(1,1,2) ≤ (6,2,2) → 满足，执行 P0

**安全序列：P2 → P1 → P0** ✅ 系统处于安全状态，可以批准新请求。

**追问变式**：若 P0 请求 `(0,2,0)`——此时 Available=(1,2,0) 且 C 已 0，则分配后 Available=(1,0,0)，P0 的 Need 变 (1,1,2)，任何进程都不满足 → **系统进入不安全状态，拒绝该请求**。

---


## 零拷贝深入：传统 IO vs mmap vs sendfile


### 传统 read + write（两次拷贝 + 四次上下文切换）

```
磁盘 → 内核缓冲区(页缓存) → 用户缓冲区 → Socket 内核缓冲区 → 网卡
  [DMA拷贝1]      [CPU拷贝1]   [CPU拷贝2]        [DMA拷贝2]
```

- **4 次拷贝**（2 次 DMA + 2 次 CPU）+ **4 次用户态/内核态切换**（read、write 各两次）


### mmap（减少一次 CPU 拷贝）

```
磁盘 → 页缓存(用户空间直接映射) → Socket 内核缓冲区 → 网卡
  [DMA拷贝1]        [CPU拷贝1]（mmap 省掉"内核→用户"拷贝）
```

- 用户态直接读页缓存，省一次 CPU 拷贝 → 3 次拷贝
- **代价**：缺页异常（首次访问映射页）、脏页回写


### sendfile（终极：仅 2 次 DMA 拷贝 + 0 次 CPU 拷贝）

```
磁盘 → 页缓存 → 网卡（SG-DMA 支持时，页缓存描述符直接交给网卡）
  [DMA拷贝1]         [DMA拷贝2]
```

- 现代 sendfile + 网卡 SG（Scatter-Gather）能力：**数据完全不经 CPU 拷贝**
- Kafka 传输、Nginx 静态文件就用它


### 对比表（背）

| 方案 | CPU 拷贝数 | DMA 拷贝数 | 上下文切换 | 适用 |
|------|-----------|-----------|-----------|------|
| 传统 read+write | 2 | 2 | 4 | 小文件、需要改数据 |
| mmap + write | 1 | 2 | 4 | 大文件、随机访问 |
| sendfile | 0 | 2 | 2 | 静态文件/网卡直传（最优） |

---


## 内存管理补充：分段、多级页表与 TLB


### 分页 vs 分段

| 维度 | 分页 | 分段 |
|------|------|------|
| 划分单位 | 固定大小页（如 4KB） | 逻辑段（代码/数据/栈） |
| 大小 | 固定（无外碎片） | 可变（有外碎片，需紧凑） |
| 由谁划分 | 系统透明划分 | 程序员/编译器按逻辑划分 |
| 地址结构 | 页号 + 页内偏移 | 段号 + 段内偏移 |
| 共享/保护 | 困难 | 方便（按段控制访问权限） |


### 多级页表为什么省内存

单级页表：4GB 虚拟地址 / 4KB 页 = 1M 页表项 × 4B = **4MB/进程**，且必须连续。

两级页表（页目录 + 页表）：只给**用到的页**分配页表 → 稀疏地址空间下可省 90%+ 内存；代价是访问一次内存要多查一级（TLB 命中可抵消）。


### TLB（Translation Lookaside Buffer）

- 页表查询的**硬件缓存**（CPU 内），保存最近用过的"虚拟页号→物理页号"
- **TLB 缺失** → 查多级页表；**进程切换**时 TLB 需刷掉/带 ASID 区分（**上下文切换成本之一**）
- TLB 命中率通常 >99%，是现代 CPU 分页性能的关键

---

**_本节验收**_：能手算 FCFS/SJF/RR 平均等待时间；会跑一遍银行家算法安全序列；能画出传统 IO vs sendfile 的拷贝路径；能说清多级页表与 TLB 解决的问题。


## 🔴 必背总结卡（进程/线程/协程 + 进程五态）

### 一句话版

- **进程**：资源分配基本单位，独立地址空间，通信靠 IPC，切换成本最高
- **线程**：CPU 调度基本单位，共享进程地址空间，靠锁/条件变量同步
- **协程**：用户态轻量调度，无内核参与，切换成本最低，适合 IO 密集

### 进程五态（背）

```text
新建 → 就绪 ⇄ 运行 → 阻塞 → 就绪 → 终止
         ↑_______________|
```
- **就绪→运行**：调度器分配 CPU；**运行→就绪**：时间片用完被抢占
- **运行→阻塞**：等待 IO/锁/事件；**阻塞→就绪**：等待的事件完成
- **面试追问**：为什么线程切换比进程切换快？→ 同进程线程共享地址空间，只需换寄存器/栈/PC，无需换页表刷 TLB

---


In [ ]:
# schedule_sim.py —— FCFS / SJF(非抢占) / RR 平均等待时间数值验证（对应手算题 P1(0,8) P2(1,4) P3(2,9)）
from collections import deque

def fcfs(jobs):
    jobs = sorted(jobs)
    t, waits = 0, []
    for a, r in jobs:
        if t < a:
            t = a
        waits.append(t - a)
        t += r
    return sum(waits) / len(waits)

def sjf_nonpreemptive(jobs):
    jobs = sorted(jobs)
    n = len(jobs)
    t = jobs[0][0]
    rest = [r for _, r in jobs]
    done = [False] * n
    waits = [0.0] * n
    for _ in range(n):
        cand = [i for i in range(n) if jobs[i][0] <= t and not done[i]]
        if not cand:
            t = min(jobs[i][0] for i in range(n) if not done[i])
            cand = [i for i in range(n) if jobs[i][0] <= t and not done[i]]
        i = min(cand, key=lambda k: rest[k])
        done[i] = True
        waits[i] = t - jobs[i][0]
        t += rest[i]
    return sum(waits) / n

def rr(jobs, q):
    jobs = sorted(jobs)
    n = len(jobs)
    rest = [r for _, r in jobs]
    comp = [0.0] * n
    done = [False] * n
    t = jobs[0][0]
    order = deque()
    finish = 0

    def enqueue_new(cur):
        for i in range(n):
            if not done[i] and jobs[i][0] <= cur and i not in order:
                order.append(i)

    enqueue_new(t)
    while finish < n:
        if not order:
            t = min(jobs[i][0] for i in range(n) if not done[i])
            enqueue_new(t)
            continue
        i = order.popleft()
        run = min(q, rest[i])
        rest[i] -= run
        t += run
        enqueue_new(t)
        if rest[i] > 0:
            order.append(i)
        else:
            done[i] = True
            comp[i] = t
            finish += 1
    # 严格口径: 等待 = 完成时间 - 到达时间 - 运行时间
    return sum(comp[i] - jobs[i][0] - jobs[i][1] for i in range(n)) / n

jobs = [(0, 8), (1, 4), (2, 9)]
print(f'FCFS 平均等待: {fcfs(jobs):.2f}   手算 ≈5.67 ✓')
print(f'SJF(非抢占) 平均等待: {sjf_nonpreemptive(jobs):.2f}')
print(f'RR(q=3) 严格口径平均等待: {rr(jobs, 3):.2f}   手算简化口径 ≈6，差异来自口径定义（面试讲相对大小即可）')


## 🟡 上下文切换成本详解（面试必答）

### 切换时到底保存什么

- **进程切换**：寄存器（通用寄存器/PC/SP）+ 页表基址（换 CR3）+ TLB 刷掉（或用 ASID 区分）+ 内核栈/用户栈
- **线程切换**（同进程）：只需换寄存器 + 栈指针 + PC，**页表不换、TLB 不刷** → 明显更便宜
- **协程切换**：纯用户态，只保存协程的局部变量/栈帧 + 指令地址，几十纳秒级

### 成本对比表（背）

| 切换类型 | 是否进内核 | 是否换页表/TLB | 相对开销 |
|---|---|---|---|
| 协程切换 | 否 | 否 | 1x |
| 同进程线程切换 | 是 | 否 | ~10x |
| 不同进程切换 | 是 | 是 | ~100x |

> **面试加分句**：高并发 IO 场景用协程（asyncio），躲开 GIL 等待且切换成本压到最低。

---


In [ ]:
# page_repl.py —— 页面置换算法缺页数对比（FIFO / LRU / OPT）
def fifo(pages, cap):
    mem, faults = [], 0
    for p in pages:
        if p not in mem:
            faults += 1
            if len(mem) == cap:
                mem.pop(0)
            mem.append(p)
    return faults

def lru(pages, cap):
    mem, faults = [], 0
    for p in pages:
        if p in mem:
            mem.remove(p)
            mem.append(p)          # 最近使用移到末尾
        else:
            faults += 1
            if len(mem) == cap:
                mem.pop(0)          # 淘汰最久未用
            mem.append(p)
    return faults

def opt(pages, cap):
    mem, faults = [], 0
    for i, p in enumerate(pages):
        if p not in mem:
            faults += 1
            if len(mem) == cap:
                def future(x):
                    return pages[i + 1:].index(x) if x in pages[i + 1:] else 10 ** 9
                far = max(mem, key=future)   # 未来最远才用到的淘汰
                mem.remove(far)
            mem.append(p)
    return faults

seq = [1, 2, 3, 4, 1, 2, 5, 1, 2, 3, 4, 5]   # 教材经典序列
for cap in (3, 4):
    print(f'容量 {cap}: FIFO 缺页 {fifo(seq, cap)}, LRU {lru(seq, cap)}, OPT {opt(seq, cap)}')
print('→ 本序列 FIFO 出现 Belady 异常：容量 3→4 缺页反而变多（9→10），说明 FIFO 缺"栈"特性')
print('  面试点: LRU/OPT 无 Belady 异常; FIFO 有')


## 🟡 Python 并发选型（GIL 场景落地，后端高频）

### GIL 是什么

- GIL（全局解释器锁）：同一时刻**只有一个线程**执行 Python 字节码
- 只影响 **CPU 密集型**：多线程无法并行，反而有锁竞争开销
- **IO 密集型不受影响**：线程在等待 IO 时释放 GIL，其他线程可运行

### 选型表（背）

| 场景 | 推荐 | 原因 |
|---|---|---|
| CPU 密集（计算） | 多进程 | 每个进程独立 GIL，真并行 |
| IO 密集（网络/文件/DB） | 多线程 / 协程 | IO 等待释放 GIL，线程切换便宜 |
| 高并发连接数 > 1 万 | 协程 asyncio | 线程有 8MB 栈上限，协程几十字节 |
| 混合场景 | 进程池 + 线程池 | 计算进进程池，IO 进线程池 |

> **易错点**：不是"多线程没用"，而是"多线程对 CPU 密集没用"。面试这样开头即可满分。

---


In [ ]:
# deadlock_detect.py —— 资源分配图成环检测（DFS 三色标记）
def detect_cycle(edges):
    """edges: [(holder, requester)]，holder 持有资源，requester 在等待"""
    g = {}
    for a, b in edges:
        g.setdefault(a, []).append(b)
    color = {}   # 0=未访问 1=在栈上 2=已完成

    def dfs(u):
        color[u] = 1
        for v in g.get(u, []):
            if color.get(v) == 1:
                return True          # 回到栈上节点 → 有环
            if color.get(v) is None and dfs(v):
                return True
        color[u] = 2
        return False

    return any(dfs(u) for u in list(g))

# P1 持 R1 等 R2；P2 持 R2 等 R1 → 循环等待 → 死锁
print('P1↔P2 互相等待(死锁):', detect_cycle([('P1', 'P2'), ('P2', 'P1')]))
print('链式等待无环:', detect_cycle([('P1', 'P2'), ('P2', 'P3')]))
print('自环(进程等自己持有的资源):', detect_cycle([('P1', 'P1')]))
print('→ 检测到环后, 解除死锁 = 打破必要条件之一: 剥夺资源/进程回滚/破坏循环等待(资源按编号序申请)')


## 🟢 补充概念：僵尸/孤儿/守护进程/写时复制

### 僵尸 vs 孤儿（必背）

- **僵尸进程**：子进程已退出，但父进程没调用 `wait()` 回收 → 进程表残留 PCB（占 PID，几乎不占资源）
  - 危害：PID 耗尽；解法：父进程 `wait()`/处理 `SIGCHLD`，或让 init 收养
- **孤儿进程**：父进程先退出，子进程被 **init (PID=1)** 收养，最终正常回收（无害）

### 写时复制 COW（fork 优化）

- fork 时子进程**共享**父进程物理页（只读映射），不复制
- 任一方**首次写入**该页时触发缺页异常，才复制私有副本 → 省内存、加速 fork
- **面试点**：为什么 fork 快？→ COW 让 fork 基本是 O(1) 元信息复制

### 可重入 vs 线程安全

- 可重入：可被中断后再次安全调用（不依赖全局/静态状态）；线程安全：多线程并发调用安全
- 不等价：线程安全不一定可重入（如用信号量保护，可能被中断后死锁）

---


## 🟡 页表项内容与 TLB 细节（虚拟内存背诵卡）

### 一个页表项包含什么

```text
| 页框号(物理页号) | 有效位 | 脏位 | 访问位 | 权限位R/W | 用户/内核位 |
```

- **有效位**：页是否在物理内存（不在 → 缺页中断）
- **脏位**：页被写过，换出时要写回磁盘
- **访问位**：供 Clock/二次机会算法选择淘汰

### TLB 命中/缺失

- TLB 是页表查询的**硬件缓存**，命中率通常 >99%，命中不访问内存
- **缺失惩罚**：完整走两级/三级页表（多 2~4 次内存访问），软件填充 TLB
- 进程切换时 TLB 失效 → **上下文切换成本的一大来源**（ASID 可避免全刷）

### 面试追问：为什么用多级页表

- 单级页表：4GB/4KB = 1M 项 × 4B = 4MB/进程 且须**连续**（稀疏地址空间浪费）
- 多级（页目录+页表）：只为**用到的页**建立页表 → 稀疏场景省 90%+ 内存
- 代价：一次访问多查 1~2 级，但 TLB 命中覆盖绝大多数访问

---


In [ ]:
# pool_size.py —— 线程/进程池大小经验公式计算器（背公式 + 自己会算）
import os

def io_threads(cores, block_ratio=0.9):
    """I/O 密集: N = 核心数 / (1 - 阻塞占比)"""
    return int(cores / (1 - block_ratio))

def cpu_processes(cores, overhead=1.1):
    """CPU 密集: N ≈ 核心数 * 1.1（留一点系统余量）"""
    return int(cores * overhead)

cores = os.cpu_count() or 4
print(f'本机 CPU 核心数: {cores}')
print(f'I/O 密集型线程池建议: {io_threads(cores)} 个线程（阻塞占比 0.9 时）')
print(f'CPU 密集型进程池建议: {cpu_processes(cores)} 个进程')
print('补充: 线程数不是越多越好——上下文切换会吃掉收益, 超线程 2x 后收益递减')


## 🔴 零拷贝背诵卡（4 → 3 → 2 次拷贝推导）

### 传统 read + write

```text
磁盘 → 页缓存 → 用户缓冲区 → Socket 内核缓冲 → 网卡
  [DMA]      [CPU]      [CPU]           [DMA]      = 2 CPU + 2 DMA, 4 次上下文切换
```
### mmap + write

```text
磁盘 → 页缓存(用户空间直接映射读) → Socket 内核缓冲 → 网卡 = 1 CPU + 2 DMA
```
- 节省"内核→用户"那次 CPU 拷贝；代价：缺页异常、脏页回写
### sendfile（SG-DMA）

```text
磁盘 → 页缓存 → 网卡 = 0 CPU + 2 DMA, 2 次上下文切换   ← 最优
```
- 网卡支持 SG（Scatter-Gather）时，页缓存描述符直接交给网卡搬运
- **应用**：Kafka 发消息、Nginx 静态文件、文件下载服务

> 面试串讲："大文件传输用 sendfile 零拷贝，Kafka 快的原因之一就是它"。

---


## 🟡 内存管理补充背诵（分段/多级页表/TLB 三连问）

### 分页 vs 分段（表格背）

| 维度 | 分页 | 分段 |
|---|---|---|
| 划分 | 固定 4KB 页 | 逻辑段(代码/数据/栈) |
| 碎片 | 页内碎片(小) | 外碎片(大，需紧凑) |
| 可见性 | 系统透明 | 程序员/编译器可见 |
| 共享/保护 | 难 | 按段控制，容易 |
| 地址结构 | 页号+偏移 | 段号+偏移 |

### 面试追问速答

- Q：为什么 32 位机器进程虚拟地址空间是 4GB？→ 2^32 = 4G 个字节
- Q：如何用虚拟内存装下比物理内存大的程序？→ 按需调页 + 页面置换，程序只需局部性
- Q：TLB 和 CPU 缓存区别？→ TLB 缓存"虚拟→物理"映射；Cache 缓存数据；缺 TLB 多访内存，缺 Cache 访磁盘
- Q：大页（HugePage）有什么用？→ 减少页表项/TLB 缺失，适合数据库缓冲池等大内存应用
